# 🌊 OceanGo: AlphaZero Self-Play Training on Google Colab

This notebook trains the **OceanGo** AlphaZero engine utilizing Google Colab's NVIDIA GPUs (T4, V100, or A100 Tensor Cores).

### Key Benefits:
- **Accelerated GPU Inference**: NVIDIA CUDA Tensor Cores dramatically speed up batched MCTS inference.
- **Direct Google Drive Persistence**: All iteration checkpoints (`policy_value_net_iter_X.pth`) save directly to your Google Drive so nothing is lost upon disconnect.
- **Built-in Dry-Run**: Includes a 10-second verification cell so you never waste compute units on misconfigurations.

## Step 1: Verify Hardware Accelerator
Ensure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> Hardware accelerator -> GPU`).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU accelerator not detected. Please switch to GPU in Runtime -> Change runtime type.")

## Step 2: Mount Google Drive
Mounting Google Drive ensures your checkpoints persist permanently even if Colab disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create dedicated checkpoint directory on your Google Drive
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/OceanGo/checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved directly to: {DRIVE_CHECKPOINT_DIR}")

## Step 3: Clone Repository & Install Dependencies

In [ ]:
# Clone repo (or pull if already present)
import os
if not os.path.isdir('/content/OceanGo'):
    !git clone https://github.com/Ocean48/OceanGo.git /content/OceanGo
else:
    !git -C /content/OceanGo pull

%cd /content/OceanGo
!pip install -r requirements.txt

## Step 4: Dry-Run Verification (Costs ~0 Compute Units)
Runs a quick 10-second end-to-end sanity check to verify CUDA inference, multiprocessing queues, and Google Drive write permissions before starting the full training.

In [ ]:
!python OceanGo.py --dry-run --save-dir /content/drive/MyDrive/OceanGo/checkpoints

## Step 5: Start Full AlphaZero Training
Launches the training pipeline. All models and checkpoints will automatically sync to your Google Drive.

- **Credit-Optimized Preset (Recommended)**: `--games 200 --sims 400` trains an iteration in ~3 to 4 hours on a T4 GPU, maximizing your monthly compute units for 30+ iterations.
- **DeepMind Paper Preset**: You can set `--games 500 --sims 800` if you want the full DeepMind paper benchmark.
- **Background Execution**: You can close the tab or turn off your computer; Colab Pro will continue running in the cloud.

In [ ]:
!python OceanGo.py --train \
    --save-dir /content/drive/MyDrive/OceanGo/checkpoints \
    --games 200 \
    --sims 400 \
    --iters 50

# Automatically disconnect and shut down the Colab GPU runtime when training finishes
# so zero Compute Units are wasted on idle time.
from google.colab import runtime
print("Training finished! Automatically disconnecting runtime to save Compute Units...")
runtime.unassign()